In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import japanize_matplotlib
import seaborn as sns
import os
import pickle
import gc


from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings("ignore")

# Pathsクラス
class Paths:
    p = "/Users/shirokoshikentaro/Desktop/Python3/house-prices-advanced-regression-techniques"
    train = p + "/train.csv"
    test = p + "/test.csv"
    sample = p + "/sample_submission.csv"

# === データ読み込み ===
print("データ読み込み中...")
train_raw = pd.read_csv(Paths.train)
test_raw = pd.read_csv(Paths.test)

# === 特徴量定義 ===
numeric_features = [
    "LotArea", "YearBuilt", "YearRemodAdd",
    "GrLivArea", "TotalBsmtSF", "1stFlrSF", "2ndFlrSF",
    "FullBath", "BedroomAbvGr", "TotRmsAbvGrd",
    "GarageCars", "GarageArea",
    "OverallQual", "OverallCond",
]

categorical_features = [
    "Neighborhood", "BldgType", "HouseStyle",
    "MSZoning", "Foundation", "GarageType"
]

features = numeric_features + categorical_features
print(f"使用する特徴量: {len(features)}個")

# === 特徴量を抽出 ===
train_features = train_raw[features].copy()
test_features = test_raw[features].copy()

# === 欠損値処理 ===
print("欠損値処理中...")
for col in numeric_features:
    train_features[col] = train_features[col].fillna(0)
    test_features[col] = test_features[col].fillna(0)

for col in categorical_features:
    train_features[col] = train_features[col].fillna("None")
    test_features[col] = test_features[col].fillna("None")

# === カテゴリ変数のエンコーディング ===
print("カテゴリ変数をエンコーディング中...")
train_encoded = {}
test_encoded = {}

for col in numeric_features:
    train_encoded[col] = train_features[col].values
    test_encoded[col] = test_features[col].values

for col in categorical_features:
    all_data = pd.concat([
        train_features[col].astype(str), 
        test_features[col].astype(str)
    ], axis=0)
    
    le = LabelEncoder()
    le.fit(all_data)
    
    train_encoded[col] = le.transform(train_features[col].astype(str))
    test_encoded[col] = le.transform(test_features[col].astype(str))

# === 新しいDataFrameを作成 ===
X_train = pd.DataFrame(train_encoded, columns=features)
X_test = pd.DataFrame(test_encoded, columns=features)

# ★★★ 対数変換（ここで確実に実行）★★★
print("\n" + "=" * 50)
print("対数変換を実行")
print("=" * 50)
y_train_original = train_raw["SalePrice"].copy()
y_train = np.log1p(y_train_original)

print(f"元のSalePrice:")
print(f"  平均: ${y_train_original.mean():,.0f}")
print(f"  範囲: ${y_train_original.min():,.0f} ~ ${y_train_original.max():,.0f}")
print(f"\n対数変換後:")
print(f"  平均: {y_train.mean():.3f}")
print(f"  範囲: {y_train.min():.3f} ~ {y_train.max():.3f}")

# 検証：対数変換が正しく適用されたか確認
if y_train.mean() > 100:
    print("\n⚠️ エラー：対数変換が失敗しています！")
    print("プログラムを中断します。")
    raise ValueError("対数変換が正しく適用されていません")
else:
    print("\n✅ 対数変換が正しく適用されました")

# === モデルパラメータ ===
params = {
    "boosting_type": "gbdt",
    "objective": "regression",
    "metric": "rmse",
    "num_leaves": 16,
    "learning_rate": 0.1,
    "n_estimators": 10000,
    "random_state": 123,
    "verbose": -1,
}

# === CV実行 ===
print("\n" + "=" * 50)
print("Cross Validation 開始")
print("=" * 50)

n_splits = 5
cv = KFold(n_splits=n_splits, shuffle=True, random_state=123)
metrics = []

for nfold, (train_index, valid_index) in enumerate(cv.split(X_train, y_train)):
    print(f"\n{'=' * 10} Fold {nfold} {'=' * 10}")
    X_tr, y_tr = X_train.iloc[train_index], y_train.iloc[train_index]
    X_val, y_val = X_train.iloc[valid_index], y_train.iloc[valid_index]

    model_fold = lgb.LGBMRegressor(**params)
    model_fold.fit(
        X_tr, y_tr,
        eval_set=[(X_tr, y_tr), (X_val, y_val)],
        eval_metric="rmse",
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=True),
            lgb.log_evaluation(0),
        ],
    )

    y_tr_pred = model_fold.predict(X_tr)
    y_val_pred = model_fold.predict(X_val)
    
    tr_rmse = np.sqrt(mean_squared_error(y_tr, y_tr_pred))
    val_rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
    
    metrics.append([nfold, tr_rmse, val_rmse])
    print(f"Train RMSE: {tr_rmse:.5f}, Valid RMSE: {val_rmse:.5f}")

# === CVスコア ===
metrics_array = np.array(metrics)
print("\n" + "=" * 50)
print("CV結果（対数スケール）")
print("=" * 50)
print("[CV] train: {:.5f}±{:.5f}, valid: {:.5f}±{:.5f}".format(
    metrics_array[:,1].mean(), metrics_array[:,1].std(),
    metrics_array[:,2].mean(), metrics_array[:,2].std(),
))
print("=" * 50)

# === 全データで最終モデルを学習 ===
print("\n全データでモデルを再学習中...")
model = lgb.LGBMRegressor(**params)
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train)],
    eval_metric="rmse",
    callbacks=[
        lgb.early_stopping(stopping_rounds=100, verbose=True),
        lgb.log_evaluation(100),
    ],
)
print("✅ 再学習完了！")

# === testデータで予測 ===
print("\n" + "=" * 50)
print("Test予測")
print("=" * 50)

y_test_pred_log = model.predict(X_test)

print(f"\n予測値（対数スケール）:")
print(f"  平均: {y_test_pred_log.mean():.3f}")
print(f"  範囲: {y_test_pred_log.min():.3f} ~ {y_test_pred_log.max():.3f}")

# 検証
if y_test_pred_log.mean() > 100:
    print("\n⚠️ 警告：予測値が対数スケールではありません")
    print("対数変換なしで進めます...")
    y_test_pred = y_test_pred_log
else:
    print("\n✅ 対数スケールの予測です")
    y_test_pred = np.expm1(y_test_pred_log)

print(f"\n最終予測価格（元のスケール）:")
print(f"  平均: ${y_test_pred.mean():,.0f}")
print(f"  範囲: ${y_test_pred.min():,.0f} ~ ${y_test_pred.max():,.0f}")

# === 提出ファイル ===
df_submit = pd.DataFrame({
    "Id": test_raw["Id"],
    "SalePrice": y_test_pred,
})

print("\n提出ファイル（最初の10行）:")
print(df_submit.head(10))
print(f"\nデータ数: {len(df_submit)}")

df_submit.to_csv("submission_log.csv", index=False)
print("\n✅ submission_log.csv を保存しました！")

データ読み込み中...
使用する特徴量: 20個
欠損値処理中...
カテゴリ変数をエンコーディング中...

対数変換を実行
元のSalePrice:
  平均: $180,921
  範囲: $34,900 ~ $755,000

対数変換後:
  平均: 12.024
  範囲: 10.460 ~ 13.534

✅ 対数変換が正しく適用されました

Cross Validation 開始

========== Fold 0 ==========
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[72]	training's rmse: 0.088505	valid_1's rmse: 0.124036
Train RMSE: 0.08850, Valid RMSE: 0.12404

========== Fold 1 ==========
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[208]	training's rmse: 0.056129	valid_1's rmse: 0.150054
Train RMSE: 0.05613, Valid RMSE: 0.15005

========== Fold 2 ==========
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[108]	training's rmse: 0.0748174	valid_1's rmse: 0.139186
Train RMSE: 0.07482, Valid RMSE: 0.13919

========== Fold 3 ==========
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is